In [9]:
!pip install requests pandas --quiet

In [10]:
import requests
import pandas as pd
from datetime import datetime


In [11]:
API_KEY = "414bea7b06156c2c90e8332ca09da109"

In [12]:
CITIES = ["Chennai", "Mumbai", "Delhi", "Bangalore", "Hyderabad"]
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"


In [15]:
def fetch_weather(city):
    params = {"q": city, "appid": API_KEY, "units": "metric"}
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        if response.status_code == 200:
            return response.json()
        else:
            msg = response.json().get("message", "unknown error")
            print(f"  [SKIP] {city}: HTTP {response.status_code} — {msg}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"  [ERROR] {city}: {e}")
        return None

raw_data = []
print("Fetching weather data...")
for city in CITIES:
    data = fetch_weather(city)
    if data:
        raw_data.append(data)
        print(f"  ✅ {city} — {data['weather'][0]['description']}")

print(f"\nFetched {len(raw_data)} / {len(CITIES)} records.")

# Stop early if nothing was fetched
if len(raw_data) == 0:
    raise ValueError(
        "\n❌ No data fetched. Possible reasons:\n"
        "  1. API key is still 'your_api_key_here' — replace it!\n"
        "  2. Key not activated yet — wait 10–15 min after signup\n"
        "  3. No internet connection in this runtime\n"
        "Fix the key in Cell 3 and re-run from Cell 3 onwards."
    )


# ── CELL 5: TRANSFORM — Parse & clean with Pandas ───────────
def parse_record(d):
    return {
        "city":             d["name"],
        "country":          d["sys"]["country"],
        "temperature_c":    round(d["main"]["temp"], 2),
        "feels_like_c":     round(d["main"]["feels_like"], 2),
        "temp_min_c":       round(d["main"]["temp_min"], 2),
        "temp_max_c":       round(d["main"]["temp_max"], 2),
        "humidity_pct":     d["main"]["humidity"],
        "pressure_hpa":     d["main"]["pressure"],
        "weather":          d["weather"][0]["main"],
        "description":      d["weather"][0]["description"].title(),
        "wind_speed_mps":   d["wind"]["speed"],
        "visibility_km":    round(d.get("visibility", 0) / 1000, 2),
        "cloudiness_pct":   d["clouds"]["all"],
        "sunrise":          datetime.utcfromtimestamp(d["sys"]["sunrise"]).strftime("%H:%M:%S"),
        "sunset":           datetime.utcfromtimestamp(d["sys"]["sunset"]).strftime("%H:%M:%S"),
        "fetched_at":       datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"),
    }

df = pd.DataFrame([parse_record(d) for d in raw_data])

print(f"DataFrame shape  : {df.shape}")
print(f"Columns          : {df.columns.tolist()}\n")

# --- Cleaning steps ---

# 1. Remove duplicate cities
before = len(df)
df.drop_duplicates(subset="city", inplace=True)
print(f"[Dedup]   Removed {before - len(df)} duplicate(s). Rows now: {len(df)}")

# 2. Drop rows where key fields are missing
df.dropna(subset=["temperature_c", "humidity_pct"], inplace=True)
print(f"[Nulls]   Rows after dropping nulls: {len(df)}")

# 3. Sanity-check value ranges
df["temperature_c"] = df["temperature_c"].clip(-90, 60)
df["humidity_pct"]  = df["humidity_pct"].clip(0, 100)
print("[Clip]    temperature_c clipped to [-90, 60]")
print("[Clip]    humidity_pct  clipped to [0, 100]")

# 4. Derived columns
df["temp_range_c"] = round(df["temp_max_c"] - df["temp_min_c"], 2)

df["heat_index"] = df["temperature_c"].apply(
    lambda t: "Hot"  if t >= 35 else
              "Warm" if t >= 25 else
              "Mild" if t >= 15 else "Cold"
)

df.reset_index(drop=True, inplace=True)

print("\n── Cleaned DataFrame ─────────────────────────────────")
print(df[["city", "temperature_c", "feels_like_c",
          "humidity_pct", "weather", "heat_index"]].to_string(index=False))


# ── CELL 6: Analytics / Summary ─────────────────────────────
print("\n── Summary Statistics ────────────────────────────────")
print(df[["temperature_c", "feels_like_c",
          "humidity_pct", "wind_speed_mps",
          "temp_range_c"]].describe().round(2).to_string())

hottest  = df.loc[df["temperature_c"].idxmax()]
coldest  = df.loc[df["temperature_c"].idxmin()]
most_hum = df.loc[df["humidity_pct"].idxmax()]
windiest = df.loc[df["wind_speed_mps"].idxmax()]

print(f"\n  🌡️  Hottest   : {hottest['city']}  ({hottest['temperature_c']}°C)")
print(f"  🥶  Coldest   : {coldest['city']}  ({coldest['temperature_c']}°C)")
print(f"  💧  Humid     : {most_hum['city']} ({most_hum['humidity_pct']}%)")
print(f"  💨  Windiest  : {windiest['city']} ({windiest['wind_speed_mps']} m/s)")

print("\n── Weather Conditions ────────────────────────────────")
print(df[["city", "weather", "description", "wind_speed_mps",
          "visibility_km", "cloudiness_pct"]].to_string(index=False))


# ── CELL 7: LOAD — Save to CSV ──────────────────────────────
output_file = "weather_data_cleaned.csv"
df.to_csv(output_file, index=False)
print(f"\n✅ Saved → '{output_file}'  ({len(df)} rows × {len(df.columns)} columns)")
print("\nPreview of saved file:")
print(pd.read_csv(output_file).to_string(index=False))

# Download the file in Colab
from google.colab import files
files.download(output_file)
print("\n📥 Download started!")

Fetching weather data...
  ✅ Chennai — haze
  ✅ Mumbai — haze
  ✅ Delhi — haze
  ✅ Bangalore — scattered clouds
  ✅ Hyderabad — haze

Fetched 5 / 5 records.
DataFrame shape  : (5, 16)
Columns          : ['city', 'country', 'temperature_c', 'feels_like_c', 'temp_min_c', 'temp_max_c', 'humidity_pct', 'pressure_hpa', 'weather', 'description', 'wind_speed_mps', 'visibility_km', 'cloudiness_pct', 'sunrise', 'sunset', 'fetched_at']

[Dedup]   Removed 0 duplicate(s). Rows now: 5
[Nulls]   Rows after dropping nulls: 5
[Clip]    temperature_c clipped to [-90, 60]
[Clip]    humidity_pct  clipped to [0, 100]

── Cleaned DataFrame ─────────────────────────────────
     city  temperature_c  feels_like_c  humidity_pct weather heat_index
  Chennai          32.18         39.14            65    Haze       Warm
   Mumbai          30.99         37.57            70    Haze       Warm
    Delhi          39.05         37.33            17    Haze        Hot
Bengaluru          25.67         26.25            7

/tmp/ipykernel_5274/3337124804.py:52: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "sunrise":          datetime.utcfromtimestamp(d["sys"]["sunrise"]).strftime("%H:%M:%S"),
/tmp/ipykernel_5274/3337124804.py:53: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "sunset":           datetime.utcfromtimestamp(d["sys"]["sunset"]).strftime("%H:%M:%S"),
/tmp/ipykernel_5274/3337124804.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "fetched_at":       datetime.utc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Download started!
